In [1]:
import torch
import numpy as np

# Load the .pth file
data = torch.load("scene0000_00.pth", map_location="cpu", weights_only=False)

# Inspect data structure
def inspect(obj, indent=0):
    prefix = "  " * indent
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, torch.Tensor):
                print(f"{prefix}{k}: Tensor {v.shape} {v.dtype}")
            elif isinstance(v, np.ndarray):
                print(f"{prefix}{k}: ndarray {v.shape} {v.dtype}")
            elif isinstance(v, (dict, list, tuple)):
                print(f"{prefix}{k}: {type(v).__name__}")
                inspect(v, indent + 1)
            else:
                print(f"{prefix}{k}: {type(v).__name__} = {repr(v)[:120]}")
    elif isinstance(obj, (list, tuple)):
        print(f"{prefix}length={len(obj)}")
        for i, item in enumerate(obj[:5]):
            print(f"{prefix}[{i}]:")
            inspect(item, indent + 1)
        if len(obj) > 5:
            print(f"{prefix}... ({len(obj) - 5} more items)")

print(f"Top-level type: {type(data).__name__}\n")
inspect(data)

Top-level type: dict

coord: ndarray (81369, 3) float32
color: ndarray (81369, 3) float32
scene_id: str = 'scene0000_00'
normal: ndarray (81369, 3) float32
semantic_gt20: ndarray (81369,) int64
semantic_gt200: ndarray (81369,) int64
instance_gt: ndarray (81369,) int64


In [2]:
# Load the .pth file
data = torch.load("scene0011_00.pth", map_location="cpu", weights_only=False)

print(f"Top-level type: {type(data).__name__}\n")
inspect(data)

Top-level type: dict

coord: ndarray (237360, 3) float32
color: ndarray (237360, 3) float32
scene_id: str = 'scene0011_00'
normal: ndarray (237360, 3) float32
semantic_gt20: ndarray (237360,) int64
semantic_gt200: ndarray (237360,) int64
instance_gt: ndarray (237360,) int64


## Point Cloud Data Structure (ScanNet)

Each `.pth` file is a dictionary representing a single indoor scene. Example:

| Field | Shape | Dtype | Description |
|---|---|---|---|
| `coord` | (N, 3) | float32 | 3D spatial coordinates (x, y, z) of each point, typically in meters |
| `color` | (N, 3) | float32 | RGB color of each point, normalized to [0, 1] |
| `normal` | (N, 3) | float32 | Surface normal vector (nx, ny, nz) — the direction the surface faces at each point |
| `semantic_gt20` | (N,) | int64 | Per-point semantic label using a 20-class taxonomy (e.g., floor, wall, chair, table) |
| `semantic_gt200` | (N,) | int64 | Per-point semantic label using a finer 200-class taxonomy (e.g., "office chair" vs "dining chair") |
| `instance_gt` | (N,) | int64 | Per-point instance ID — distinguishes individual object instances within the same semantic class |
| `scene_id` | scalar | str | Scene identifier (e.g., `'scene0000_00'`) used for tracking and logging |

**Notes:**
- `N` varies per scene (e.g., 81,369 for scene0000_00, 237,360 for scene0011_00)
- `gt` suffix = ground truth annotations, used for training/evaluating semantic and instance segmentation models
- `semantic_gt20` groups points by object category; `instance_gt` further separates individual objects (e.g., 3 chairs each get a unique instance ID)

In [3]:
print(data.keys())

dict_keys(['coord', 'color', 'scene_id', 'normal', 'semantic_gt20', 'semantic_gt200', 'instance_gt'])


In [4]:
print(data['coord'])

[[2.5091114  0.4083811  0.14877559]
 [2.5156426  0.4059527  0.14168811]
 [2.5073788  0.4145141  0.14327997]
 ...
 [6.7792506  8.151305   0.07218327]
 [6.6933384  7.544996   0.03857339]
 [6.687325   7.5214887  0.03800958]]


In [5]:
print(data['color'])

[[35. 33. 38.]
 [34. 32. 39.]
 [40. 35. 43.]
 ...
 [42. 33. 23.]
 [95. 74. 43.]
 [95. 69. 43.]]


In [6]:
print(data['scene_id'])

scene0011_00


In [7]:
print(data['normal'])

[[ 0.19109516  0.92176497  0.3371149 ]
 [ 0.35799062  0.9311391   0.06885417]
 [ 0.59007823 -0.42843354  0.684249  ]
 ...
 [-0.12758346 -0.25142902  0.95939577]
 [ 0.41444626 -0.08228706  0.9063321 ]
 [-0.01580104 -0.05262993  0.9984768 ]]


In [8]:
print(data['semantic_gt20'])

[14 14 14 ...  1  1  1]


In [9]:
print(data['semantic_gt200'])

[23 23 23 ...  2  2  2]


In [10]:
print(data['instance_gt'])

[27 27 27 ...  0  0  0]


In [11]:
import numpy as np

# Load super_points binary file
sp = np.fromfile("super_points/scene0000_00.bin", dtype=np.int64)

print(f"Shape: {sp.shape}")
print(f"Dtype: {sp.dtype}")
print(f"Unique values: {len(np.unique(sp))}")
print(f"Value range: [{sp.min()}, {sp.max()}]")
print(f"\nFirst 20 values: {sp[:20]}")
print(f"\nValue counts (top 10):")
unique, counts = np.unique(sp, return_counts=True)
top_idx = np.argsort(-counts)[:10]
for i in top_idx:
    print(f"  superpoint {unique[i]}: {counts[i]} points")

Shape: (81369,)
Dtype: int64
Unique values: 667
Value range: [0, 666]

First 20 values: [ 54  54  54  54  57  54  54   0  54  54   0 164   0  56  56  55  55  56
  56  56]

Value counts (top 10):
  superpoint 164: 8075 points
  superpoint 506: 1527 points
  superpoint 408: 1385 points
  superpoint 130: 1273 points
  superpoint 416: 1163 points
  superpoint 191: 1102 points
  superpoint 605: 842 points
  superpoint 596: 770 points
  superpoint 135: 749 points
  superpoint 424: 721 points


In [12]:
# Load the .pth file
data = torch.load("scene0000_00.pth", map_location="cpu", weights_only=False)

In [15]:
print(data["instance_gt"][:20])

[ 5  5  5  5 -1  5  5 -1  5  5 -1  9 -1 57 57  5  5 57 57 57]


In [16]:
print(data["semantic_gt200"][:20])

[18 18 18 18 -1 18 18 -1 18 18 -1  2 -1 98 98 18 18 98 98 98]


## Superpoints

### What are superpoints?

Superpoints are **over-segmented clusters** of the point cloud — groups of nearby points with similar geometry (normals, spatial proximity) and appearance (color). They are analogous to "superpixels" in 2D image processing.

Each `.bin` file in `super_points/` contains a flat array of `int64` superpoint IDs, one per point:
- **Shape**: `(N,)` where N matches the point count in the corresponding `.pth` file
- **Example**: `scene0000_00.bin` has 81,369 values with 667 unique superpoints (IDs 0–666)
- Each superpoint is a small surface patch (e.g., part of a wall, a section of a chair seat, a floor region)

### Superpoint ID vs `instance_gt`

| | Superpoint ID | `instance_gt` |
|---|---|---|
| **Source** | Algorithm (over-segmentation) | Human annotation |
| **Granularity** | Fine — one object = many superpoints | Coarse — one object = one instance |
| **Purpose** | Mid-level representation for graph construction, feature aggregation, downsampling | Ground truth for instance segmentation evaluation |

**Relationship**: a single instance contains many superpoints, but one superpoint typically belongs to only one instance. For example, a chair might be split into 5–10 superpoints (seat, back, legs) that all share the same `instance_gt`.

### Why superpoints are useful
- **Graph construction**: superpoints become nodes in a scene graph, with edges between adjacent superpoints (core idea of SuperPoint Graph methods)
- **Efficiency**: operating on hundreds of superpoints instead of tens of thousands of raw points
- **Feature aggregation**: pool per-point features (color, geometry) into one descriptor per superpoint